# Hierarchical classification

**Difficulty:** Intermediate

Walk a deep taxonomy to its correct leaf with one `Choice` per node, level by level. Committing to the top pick at every step (greedy) can't recover from an early mistake; keeping a few plausible paths alive (beam search) lets deeper evidence repair an ambiguous early decision.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source (trimmed):</strong> docs.typesafe.ai/cookbooks/hierarchical_classification (click to expand)</summary>

# Hierarchical classification

> Classifies documents through deep patent, retail product, biomedical, and source-code hierarchies using parallel beam search over TypeSafe Choice probabilities.

A lot of data exists as structured hierarchies: taxonomies, filesystems, website structures, codebases, org charts, biological ontologies. Classifying into one is a perfect fit for `Choice` — classify the document at each node, starting at the root, and proceed to the next most-probable node until reaching a leaf.

## Two methods

- **Greedy search:** choose the highest-probability child, discard every alternative. One early mistake cannot be recovered.
- **Beam search:** retain `K` plausible paths, classify every frontier in parallel (extra exploration costs little wall-clock time, since paths run as parallel questions). The leaf of the path with the highest **geometric-mean edge probability** — `product(edge_probabilities) ** (1 / decisions)`, length-normalized so shallow and deep leaves compare fairly — is the final classification.

A `separation` ratio (top path's geometric mean over its nearest rival's) is a useful diagnostic even though it isn't used for pruning: near `1x` means the classification was genuinely ambiguous; a large ratio means clear separation.

## Results across four real hierarchies

Four real hierarchies (CPC patents, Shopify product categories, MeSH biomedical subjects, and TypeSafe's own cookbook-repository file structure), each with a known expected leaf:

| Hierarchy | Expected leaf | Greedy leaf | Beam K=3 leaf | Greedy correct | Beam correct |
| --- | --- | --- | --- | --- | --- |
| CPC patents | A01K31/12 Perches for poultry or birds | E99Z99/00 Subject matter not otherwise provided for | A01K31/12 Perches for poultry or birds | no | yes |
| Shopify products | Cat Window Beds & Perches | Pet Chairs | Cat Window Beds & Perches | no | yes |
| MeSH biomedical subjects | Crohn Disease | Crohn Disease | Crohn Disease | yes | yes |
| CookSafe files | retrievers.py | retrievers.py | retrievers.py | yes | yes |

**Beam search matched 4 of 4 expected leaves; greedy matched 2 of 4.** On the two hard cases (patents and products), keeping just 3 paths alive instead of 1 recovered the correct leaf — an early, ambiguous top-level choice that greedy locked itself into got corrected once deeper, more specific evidence became available.

> The full cookbook builds real trees from CPC 2026.05, Shopify's 2026-02 product taxonomy, MeSH 2026, and its own repository, and renders the full traversal as colored diagrams (orange for the greedy route, green for the winning beam route, purple for other retained paths, dashed for pruned ones) — trimmed here to a small hand-built taxonomy. Read the full cookbook at [docs.typesafe.ai/cookbooks/hierarchical_classification](https://docs.typesafe.ai/cookbooks/hierarchical_classification) any time.

</details>

## The core idea

The same taxonomy-walking idea from lesson 2's embedded Choice source (a bottle that could plausibly sit under `Sporting Goods > Cycling` or `Home & Kitchen > Drinkware`) — one `Choice` per level, each level's options being that node's children, each answer's `probabilities` telling you not just the winner but how contested the decision was. Greedy search is what you'd naturally reach for: pick the best department, then the best category within it. Beam search is the fix once you notice greedy has no way back — by keeping the top `K` departments alive and asking the second-level question for *all* of them in parallel, a strong signal two levels down can outweigh a weak signal one level up, at the cost of `K` times the calls instead of 1.

In [ ]:
import { client } from "../../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const TAXONOMY: Record<string, string[]> = {
  "Sporting Goods": ["Bike Bottles & Cages", "Yoga Mats", "Resistance Bands", "Helmets"],
  "Home & Kitchen": ["Water Bottles", "Travel Mugs", "Pots & Pans", "Bakeware"],
  "Baby & Toddler": ["Sippy Cups", "Bottle Warmers", "Bibs"],
};

// The same deliberately ambiguous listing from lesson 2's embedded Choice source.
const listing = "32oz plastic bottle with a flip straw lid. Fits most bike cages.";

async function askChoice(
  state: string,
  options: string[],
  question: string,
): Promise<Record<string, number>> {
  const criteria = Object.fromEntries(options.map((o) => [o, null]));
  const response = await client.systemOne({
    state,
    questions: { pick: choice(question, criteria) },
  });
  return response.answers.pick.probabilities;
}

function argmax(probs: Record<string, number>): string {
  return Object.entries(probs).reduce((best, cur) => (cur[1] > best[1] ? cur : best))[0];
}

// --- Greedy: commit to the top department, never look back ---
const deptProbs = await askChoice(
  listing,
  Object.keys(TAXONOMY),
  "Which top-level department does this product belong to?",
);
const greedyDept = argmax(deptProbs);
const greedyLeafProbs = await askChoice(
  listing,
  TAXONOMY[greedyDept],
  "Which specific product type is this?",
);
const greedyLeaf = argmax(greedyLeafProbs);
console.log(
  "GREEDY:",
  greedyDept,
  "->",
  greedyLeaf,
  `(dept p=${deptProbs[greedyDept].toFixed(2)}, leaf p=${greedyLeafProbs[greedyLeaf].toFixed(2)})`,
);

// --- Beam search (K=2): keep the top 2 departments alive, let deeper evidence decide ---
const K = 2;
const topDepartments = Object.entries(deptProbs)
  .sort((a, b) => b[1] - a[1])
  .slice(0, K)
  .map(([name]) => name);

const beamResults = await Promise.all(
  topDepartments.map(async (dept) => {
    const leafProbs = await askChoice(listing, TAXONOMY[dept], "Which specific product type is this?");
    const leaf = argmax(leafProbs);
    // Length-normalized geometric mean over the path's edge probabilities.
    const pathScore = Math.exp((Math.log(deptProbs[dept]) + Math.log(leafProbs[leaf])) / 2);
    return { dept, leaf, pathScore, deptProb: deptProbs[dept], leafProb: leafProbs[leaf] };
  }),
);

beamResults.sort((a, b) => b.pathScore - a.pathScore);
console.log("BEAM candidates:", beamResults);
console.log(
  "BEAM winner:",
  beamResults[0].dept,
  "->",
  beamResults[0].leaf,
  `(path score=${beamResults[0].pathScore.toFixed(3)})`,
);

## What just happened

The listing genuinely straddles two departments: "bottle" and "flip straw lid" point at `Home & Kitchen > Water Bottles`, but "fits most bike cages" is a strong, specific signal for `Sporting Goods > Bike Bottles & Cages`. If the top-level `Choice` happens to favor `Home & Kitchen` even slightly, greedy locks in there and never gets to weigh the bike-cage evidence against `Water Bottles` specifically. Beam search keeps both departments alive and asks the leaf-level question for *both*, so the "fits most bike cages" phrase gets to compete directly against `Water Bottles`, `Travel Mugs`, and every other real alternative — exactly the mechanism that flipped the CPC patents and Shopify cases in the primary source's own results.

**Try it yourself:** predict, before running, whether `GREEDY` and `BEAM winner` land on the same department. Then reduce `K` to 1 (beam search degenerates to greedy) and confirm the two methods now agree by construction.

## Related

- `lessons/core-concepts/0002-choice.ipynb` — the "walking a taxonomy" section of the embedded Choice source, the direct ancestor of this cookbook's listing example
- `cookbooks/how-to/re-ranking.ipynb` — another use of independent scores to pick the best of several candidates, without the level-by-level tree structure
- `lessons/patterns/0006-speculative-fan-out.ipynb` — why beam search's extra paths cost little wall-clock time: each level's candidate paths are independent parallel calls, not a sequential chain

Questions? Ask your agent — it has the same research this cookbook was built from, in `research/typesafe-ai.md`.